# Trigger Management via az CLI

Manage triggers using the `az connectorgateway` CLI extension.
This is a follow-on lab — run `01-trigger-getting-started.ipynb` first
to create the gateway, connection, and sandbox.

## Prerequisites
- Azure CLI [signed in](https://learn.microsoft.com/cli/azure/authenticate-azure-cli-interactively)
- Connector Gateway CLI extension: `az extension add --source <path-to-az_cli_connectorgateway-*.whl>`
- Gateway + connection from lab 01

### 0. Initialize

In [ ]:
import os, sys, subprocess, json

_SHELL = sys.platform == 'win32'

lab_name = os.path.basename(os.path.dirname(os.path.abspath(
    globals().get('__vsc_ipynb_file__', os.path.join(os.getcwd(), '__file__')))))

resource_group = f'lab-{lab_name}'
gateway = f'{lab_name}-gw'

result = subprocess.run(['az', 'connectorgateway', 'trigger', '--help'],
    capture_output=True, text=True, shell=_SHELL)
print(result.stdout[:500] if result.returncode == 0 else f'Error: {result.stderr}')

### 1. List triggers

In [ ]:
result = subprocess.run(
    ['az', 'connectorgateway', 'trigger', 'list', '-g', resource_group, '--gateway', gateway, '-o', 'json'],
    capture_output=True, text=True, shell=_SHELL)
triggers = json.loads(result.stdout) if result.returncode == 0 else []
print(f'{len(triggers)} trigger(s):')
for t in triggers:
    print(f'  {t["name"]}: {t.get("properties", {}).get("state", "?")}')

### 2. Discover trigger operations

In [ ]:
result = subprocess.run(
    ['az', 'connectorgateway', 'trigger', 'operations', 'list', '-g', resource_group,
     '--gateway', gateway, '--connector-type', 'office365', '-o', 'json'],
    capture_output=True, text=True, shell=_SHELL)
if result.returncode == 0:
    ops = json.loads(result.stdout)
    for op in ops:
        print(f'  {op["operationId"]}: {op.get("summary", "")}')
else:
    print(f'Error: {result.stderr}')

### 3. Create a trigger

In [ ]:
result = subprocess.run([
    'az', 'connectorgateway', 'trigger', 'create',
    '-g', resource_group,
    '--gateway', gateway,
    '-n', 'cli-trigger',
    '--connector-name', 'office365',
    '--connection-name', 'o365-trigger-conn',
    '--operation-name', 'OnNewEmail',
    '--sandbox-id', 'my-sandbox-id',
    '-s', 'my-sandbox-group',
    '--command', 'echo trigger fired >> /tmp/trigger.log',
    '-o', 'json'
], capture_output=True, text=True, shell=_SHELL)
if result.returncode == 0:
    trigger = json.loads(result.stdout)
    print(f'Created: {trigger["name"]} ({trigger.get("properties", {}).get("state", "?")})')
else:
    print(f'Error: {result.stderr}')

### 4. Enable / disable trigger

In [ ]:
result = subprocess.run(
    ['az', 'connectorgateway', 'trigger', 'disable', '-g', resource_group, '--gateway', gateway, '-n', 'cli-trigger', '-o', 'json'],
    capture_output=True, text=True, shell=_SHELL)
if result.returncode == 0:
    print(f'Disabled: {json.loads(result.stdout).get("properties", {}).get("state", "?")}')

result = subprocess.run(
    ['az', 'connectorgateway', 'trigger', 'enable', '-g', resource_group, '--gateway', gateway, '-n', 'cli-trigger', '-o', 'json'],
    capture_output=True, text=True, shell=_SHELL)
if result.returncode == 0:
    print(f'Enabled: {json.loads(result.stdout).get("properties", {}).get("state", "?")}')

### 5. Delete trigger

In [ ]:
result = subprocess.run(
    ['az', 'connectorgateway', 'trigger', 'delete', '-g', resource_group, '--gateway', gateway, '-n', 'cli-trigger', '--yes'],
    capture_output=True, text=True, shell=_SHELL)
print('Deleted' if result.returncode == 0 else f'Error: {result.stderr}')